In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
df = pd.read_csv("./dataset/winequality-red.csv")

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
def graficar_distribucion_target():
  # Calcular frecuencias absolutas y relativas
  quality_counts = df["quality"].value_counts().sort_index()
  quality_perc = (quality_counts / quality_counts.sum()) * 100

  # Gráfico de barras con dos ejes (conteo absoluto y %)
  fig, ax1 = plt.subplots(figsize=(8,5))

  # Barras (conteo absoluto)
  bars = ax1.bar(quality_counts.index, quality_counts.values, color="skyblue")
  ax1.set_xlabel("Calidad del vino (quality)")
  ax1.set_ylabel("Número de muestras", color="blue")
  ax1.tick_params(axis="y", labelcolor="blue")

  # Segundo eje Y para porcentajes
  ax2 = ax1.twinx()
  ax2.plot(quality_counts.index, quality_perc.values, color="red", marker="o", linewidth=2)
  ax2.set_ylabel("Porcentaje (%)", color="red")
  ax2.tick_params(axis="y", labelcolor="red")

  # Títulos y ajustes
  plt.title("Distribución del target 'quality' (conteo absoluto y %)")
  plt.xticks(quality_counts.index)

  # Mostrar valores encima de cada barra
  for bar, perc in zip(bars, quality_perc.values):
      height = bar.get_height()
      ax1.text(bar.get_x() + bar.get_width()/2, height+10,
              f"{int(height)}\n({perc:.1f}%)",
              ha="center", va="bottom", fontsize=8)

  plt.tight_layout()
  plt.show()


graficar_distribucion_target()

In [ ]:
def graficar_outliers():
  features = df.drop('quality', axis=1).columns

  # Crear box plots para identificar valores atípicos
  fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(18, 15))
  axes = axes.flatten()

  for i, col in enumerate(features):
      sns.boxplot(x=df[col], ax=axes[i], color='lightblue')
      axes[i].set_title(f'Box Plot de {col}')

  plt.tight_layout()
  plt.show()


graficar_outliers()

In [ ]:
test_size = 0.3

X = df.drop("quality", axis=1)
y = df["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    test_size=test_size,
    stratify=y
)

In [ ]:
print("Tamaño X_train:", X_train.shape)
print("Tamaño X_test:", X_test.shape)
print("Tamaño y_train:", y_train.shape)
print("Tamaño y_test:", y_test.shape)
print("Distribución en train:\n", y_train.value_counts(normalize=True))
print("Distribución en test:\n", y_test.value_counts(normalize=True))

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

In [ ]:
classes

In [ ]:
class_weights

In [ ]:
def Random_Forest(X_train, X_test, y_train, y_test, classes, class_weights):
  print("RANDOM FOREST")
  print("Tamaño X_train:", X_train.shape)
  print("Tamaño X_test:", X_test.shape)
  print("Tamaño y_train:", y_train.shape)
  print("Tamaño y_test:", y_test.shape)
  print("classes:", classes)
  print("class_weights:", class_weights)

In [ ]:
Random_Forest(X_train, X_test, y_train, y_test, classes, class_weights)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Creás una instancia del modelo
# podés ajustar los hiperparámetros, como n_estimators (número de árboles)
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight=class_weights)

# 2. Entrenás el modelo
modelo_rf.fit(X_train, y_train)

In [ ]:
# 3. Hacés las predicciones
y_pred_rf = modelo_rf.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# 4. Evaluás el modelo
print("Accuracy del Random Forest:", accuracy_score(y_test, y_pred_rf))
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred_rf))

# Matriz de confusión para visualizar mejor el rendimiento
mat_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(mat_rf, annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de Confusión - Random Forest")
plt.ylabel("Etiqueta Verdadera")
plt.xlabel("Etiqueta Predicha")
plt.show()

In [ ]:
!pip install imbalanced-learn --user

In [ ]:
print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
print("\nPrimeras 5 filas de X:")
print(X.head())
print("\nPrimeras 5 filas de y:")
print(y.head())

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter


# 1. Define el oversampler (esto está bien)
over = SMOTE(sampling_strategy={3: 150, 4: 300, 8: 150})

# 2. Define el undersampler con los números corregidos
under = RandomUnderSampler(sampling_strategy={5: 470, 6: 440}) # <-- NÚMEROS AJUSTADOS

# 3. Crea la pipeline
pipeline = Pipeline(steps=[('o', over), ('u', under)])

# 4. Aplica la pipeline (ahora funcionará)
X_train_resampled, y_train_resampled = pipeline.fit_resample(X_train, y_train)

# Muestra el resultado para verificar
print(Counter(y_train_resampled))



In [ ]:
modelo_rf.fit(X_train_resampled, y_train_resampled)
y_pred_rf = modelo_rf.predict(X_test)
# 4. Evaluás el modelo
print("Accuracy del Random Forest:", accuracy_score(y_test, y_pred_rf))
print("F1 Macro:", f1_score(y_test, y_pred_rf, average='macro'))
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred_rf))

# Matriz de confusión para visualizar mejor el rendimiento
mat_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(mat_rf, annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de Confusión - Random Forest")
plt.ylabel("Etiqueta Verdadera")
plt.xlabel("Etiqueta Predicha")
plt.show()

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Define el espacio de parámetros que quieres explorar
param_grid = {
    'n_estimators': [100, 200, 300],      # Número de árboles
    'max_depth': [10, 20, 30, None],      # Profundidad máxima de los árboles
    'min_samples_split': [2, 5, 10],      # Mínimo de muestras para dividir un nodo
    'min_samples_leaf': [1, 2, 4],        # Mínimo de muestras por hoja
}

# 2. Crea el modelo base
rf = RandomForestClassifier(random_state=42)

# 3. Configura la búsqueda aleatoria
rf_random = RandomizedSearchCV(estimator=rf, param_distributions=param_grid,
                               n_iter=50,       # Número de combinaciones a probar
                               cv=3,            # Número de folds en la validación cruzada
                               verbose=2,
                               random_state=42,
                               n_jobs=-1)       # Usa todos los procesadores

# 4. Ejecuta la búsqueda sobre los datos (idealmente los remuestreados con SMOTE)
rf_random.fit(X_train_resampled, y_train_resampled)

# 5. Muestra los mejores parámetros encontrados
print("Mejores parámetros encontrados:", rf_random.best_params_)

# El mejor modelo ya está entrenado y listo para usar
best_model = rf_random.best_estimator_

In [ ]:
y_pred_rf = best_model.predict(X_test)
# 4. Evaluás el modelo
print("Accuracy del Random Forest:", accuracy_score(y_test, y_pred_rf))
print("F1 Macro:", f1_score(y_test, y_pred_rf, average='macro'))
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred_rf))

# Matriz de confusión para visualizar mejor el rendimiento
mat_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(mat_rf, annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de Confusión - Random Forest")
plt.ylabel("Etiqueta Verdadera")
plt.xlabel("Etiqueta Predicha")
plt.show()

In [ ]:
# 'best_model' ya fue creado en tu celda anterior
from sklearn.metrics import classification_report

# Hacé las predicciones sobre los datos de prueba
y_pred_test = best_model.predict(X_test)

# Imprimí el reporte de clasificación para ver el rendimiento real
print("--- RENDIMIENTO EN EL CONJUNTO DE PRUEBA ---")
print(classification_report(y_test, y_pred_test))

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Cargar y preparar los datos
# Asegurate de que la ruta a tu archivo sea la correcta
df = pd.read_csv('./dataset/winequality-red.csv')
X = df.drop('quality', axis=1)
y = df['quality']

# 2. Escalar los datos (¡Paso crucial para PCA!)
# PCA es sensible a la escala de las variables, por lo que es necesario estandarizarlas primero.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Aplicar PCA
# Reducimos las 11 dimensiones a solo 2 componentes principales.
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 4. Crear un nuevo DataFrame para graficar
# Juntamos los 2 componentes de PCA con la etiqueta de calidad original.
df_pca = pd.DataFrame(data=X_pca, columns=['Componente Principal 1', 'Componente Principal 2'])
df_pca['quality'] = y

# 5. Crear el gráfico
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='Componente Principal 1',
    y='Componente Principal 2',
    hue='quality',          # Colorear los puntos según la calidad del vino
    data=df_pca,
    palette='viridis',      # Podés cambiar la paleta de colores
    s=80,                   # Tamaño de los puntos
    alpha=0.8               # Transparencia de los puntos
)

plt.title('Distribución de Vinos por Calidad (PCA)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend(title='Calidad del Vino')
plt.grid(True)
plt.show()